## Load the dataset

In [1]:
import os
import glob
import numpy as np
from sklearn.model_selection import train_test_split
from PIL import Image
import cv2
import imutils

In [2]:
def crop_img(img):
    """
    Finds the extreme points on the image and crops the rectangular out of them
    """
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    gray = cv2.GaussianBlur(gray, (3, 3), 0)

    # threshold the image, then perform a series of erosions +
    # dilations to remove any small regions of noise
    thresh = cv2.threshold(gray, 45, 255, cv2.THRESH_BINARY)[1]
    thresh = cv2.erode(thresh, None, iterations=2)
    thresh = cv2.dilate(thresh, None, iterations=2)

    # find contours in thresholded image, then grab the largest one
    cnts = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cnts = imutils.grab_contours(cnts)
    c = max(cnts, key=cv2.contourArea)

    # find the extreme points
    extLeft = tuple(c[c[:, :, 0].argmin()][0])
    extRight = tuple(c[c[:, :, 0].argmax()][0])
    extTop = tuple(c[c[:, :, 1].argmin()][0])
    extBot = tuple(c[c[:, :, 1].argmax()][0])
    ADD_PIXELS = 0
    new_img = img[extTop[1]-ADD_PIXELS:extBot[1]+ADD_PIXELS, extLeft[0]-ADD_PIXELS:extRight[0]+ADD_PIXELS].copy()
    
    return new_img

In [3]:
import warnings
warnings.filterwarnings('ignore')

# Or set environment variable before imports
import os
os.environ['PYTHONWARNINGS'] = 'ignore'

In [4]:
# from sklearn.datasets import fetch_openml

# X, y = fetch_openml("mnist_784", version=1, return_X_y=True, as_frame=False)

base_dir = 'brain_mri_dataset/Training'
selected_folders = ['glioma','meningioma','notumor','pituitary']

X = []
y = []

for folder in selected_folders:
    folder_path = os.path.join(base_dir, folder)
    image_paths = sorted(glob.glob(os.path.join(folder_path, '*.jpg')))
    
    for img_path in image_paths:
        # Load image as RGB for crop_img function
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Apply preprocessing: crop to remove black borders
        img_cropped = crop_img(img)
        # Convert to grayscale and PIL Image for resizing
        img_gray = cv2.cvtColor(img_cropped, cv2.COLOR_RGB2GRAY)
        img_pil = Image.fromarray(img_gray)
        img_resized = img_pil.resize((128, 128))
        X.append(np.array(img_resized))
        y.append(folder)


In [5]:
from gtda.plotting import plot_heatmap
sample_image = X[0]
plot_heatmap(sample_image)


### Create train and test sets

In [6]:
X = np.array(X)
y = np.array(y)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train set: {X_train.shape}, Test set: {X_test.shape}')
print(f'Train labels: {len(y_train)}, Test labels: {len(y_test)}')


Train set: (4569, 128, 128), Test set: (1143, 128, 128)
Train labels: 4569, Test labels: 1143


## From pixels to topological features

### Binarize the image

In ``giotto-tda``, filtrations of cubical complexes are built from _binary images_ consisting of only black and white pixels. We can convert our greyscale image to binary by applying a threshold on each pixel value via the ``Binarizer`` transformer:

In [7]:
from gtda.images import Binarizer

# Pick out index of first glioma image
im8_idx = np.flatnonzero(y_train == "glioma")[1]
# Reshape to (n_samples, n_pixels_x, n_pixels_y) format
im8 = X_train[im8_idx][None, :, :]

binarizer = Binarizer(threshold=0.3)
im8_binarized = binarizer.fit_transform(im8)

binarizer.plot(im8_binarized)

### From binary image to filtration

In [8]:
from gtda.images import RadialFiltration

radial_filtration = RadialFiltration(center=np.array([64,64]))
im8_filtration = radial_filtration.fit_transform(im8_binarized)

radial_filtration.plot(im8_filtration, colorscale="jet")

We can see from the resulting plot that we've effectively transformed our binary image into a greyscale one, where the pixel values increase as we move from the upper-right to bottom-left of the image! These pixel values can be used to define a filtration of cubical complexes $\{K_i\}_{i\in \mathrm{Im}(I)}$, where $K_i$ contains all pixels with value less than the $i$th smallest pixel value in the greyscale image. In other words, $K_i$ is the $i$th sublevel set of the image's cubical complex $K$.

In [9]:
from gtda.images import DensityFiltration

density_filtration = DensityFiltration(radius=3, n_jobs=-1)
im8_density = density_filtration.fit_transform(im8_binarized)

density_filtration.plot(im8_density, colorscale="jet")

### From filtration to persistence diagram

Given a greyscale filtration it is straightforward to calculate the corresponding persistence diagram. In ``giotto-tda`` we make use of the ``CubicalPersistence`` transformer which is the cubical analogue to simplicial transformers like ``VietorisRipsPersistence``:

In [10]:
from gtda.homology import CubicalPersistence

cubical_persistence = CubicalPersistence(n_jobs=-1)
im8_cubical = cubical_persistence.fit_transform(im8_filtration)

cubical_persistence.plot(im8_cubical)

In [11]:
from gtda.diagrams import Scaler

scaler = Scaler()
im8_scaled = scaler.fit_transform(im8_cubical)

scaler.plot(im8_scaled)

### From persistence diagram to representation

The final step is to define a vectorial representation of the persistence diagram that can be used to obtain machine learning features. Following our example from Figure 2, we convolve our persistence diagram with a Gaussian kernel and symmetrize along the main diagonal, a procedure achieved via the [``HeatKernel``](https://giotto-ai.github.io/gtda-docs/latest/modules/generated/diagrams/representations/gtda.diagrams.HeatKernel.html#gtda.diagrams.HeatKernel) transformer:

In [12]:
from gtda.diagrams import HeatKernel

heat = HeatKernel(sigma=.15, n_bins=60, n_jobs=-1)
im8_heat = heat.fit_transform(im8_scaled)

# Visualise the heat kernel for H1
heat.plot(im8_heat, homology_dimension_idx=1, colorscale='jet')

### Combining all steps as a single pipeline


In [14]:
from sklearn.pipeline import Pipeline
from gtda.diagrams import Amplitude

steps = [
    ("binarizer", Binarizer(threshold=0.3)),
    ("filtration", RadialFiltration(center=np.array([64,64]))),
    ("diagram", CubicalPersistence()),
    ("rescaling", Scaler()),
    ("amplitude", Amplitude(metric="heat", metric_params={'sigma':0.15, 'n_bins':60}))
]

heat_pipeline = Pipeline(steps)

In [15]:
im8_pipeline = heat_pipeline.fit_transform(im8)
im8_pipeline

array([[ 99.98942104, 145.69606529]])

In the final step we've used the [``Amplitude``](https://giotto-ai.github.io/gtda-docs/latest/modules/generated/diagrams/features/gtda.diagrams.Amplitude.html) transformer to "vectorize" the persistence diagram via the heat kernel method above. In our example, this produces a vector of amplitudes $\mathbf{a} = (a_0, a_1)$ where each amplitude $a_i$ corresponds to a given homology dimension in the persistence diagram. By extracting these feature vectors from each image, we can feed them into a machine learning classifier – let's tackle this in the next section!

## Building a full-blown feature extraction pipeline

In [ ]:
from sklearn.pipeline import make_pipeline, make_union
from gtda.diagrams import PersistenceEntropy
from gtda.images import DensityFiltration


center_list = [
    [64, 27],     
    [27, 64],    
    [64, 64],     
    [91, 64],    
    [64, 91],     
    [27, 27],  
    [27, 91],    
    [91, 27],    
    [91, 91],  
]

# Creating a list of all filtration transformer, we will be applying
filtration_list = [
    RadialFiltration(center=np.array(center), n_jobs=-1) 
    for center in center_list
] + [DensityFiltration(radius=3, n_jobs=-1)] 
# Add height filyration as well  

# Creating the diagram generation pipeline
diagram_steps = [
    [
        Binarizer(threshold=0.3, n_jobs=-1),  
        filtration,
        CubicalPersistence(n_jobs=-1),
        Scaler(n_jobs=-1),
    ]
    for filtration in filtration_list
]

# Listing metrics to extract diagram amplitudes
metric_list = [
    {"metric": "bottleneck", "metric_params": {}},
    {"metric": "wasserstein", "metric_params": {"p": 1}},
    {"metric": "wasserstein", "metric_params": {"p": 2}},
    {"metric": "landscape", "metric_params": {"p": 1, "n_layers": 1, "n_bins": 100}},
    {"metric": "landscape", "metric_params": {"p": 1, "n_layers": 2, "n_bins": 100}},
    {"metric": "landscape", "metric_params": {"p": 2, "n_layers": 1, "n_bins": 100}},
    {"metric": "landscape", "metric_params": {"p": 2, "n_layers": 2, "n_bins": 100}},
    {"metric": "betti", "metric_params": {"p": 1, "n_bins": 100}},
    {"metric": "betti", "metric_params": {"p": 2, "n_bins": 100}},
    {"metric": "heat", "metric_params": {"p": 1, "sigma": 1.6, "n_bins": 100}},
    {"metric": "heat", "metric_params": {"p": 1, "sigma": 3.2, "n_bins": 100}},
    {"metric": "heat", "metric_params": {"p": 2, "sigma": 1.6, "n_bins": 100}},
    {"metric": "heat", "metric_params": {"p": 2, "sigma": 3.2, "n_bins": 100}},
]

feature_union = make_union(
    *[PersistenceEntropy(nan_fill_value=-1)]
    + [Amplitude(**metric, n_jobs=-1) for metric in metric_list]
)

tda_union = make_union(
    *[make_pipeline(*diagram_step, feature_union) for diagram_step in diagram_steps],
    n_jobs=-1
)

which can be visualised using ``scikit-learn``'s nifty [HTML feature](https://scikit-learn.org/stable/modules/compose.html#visualizing-composite-estimators):

In [17]:
from sklearn import set_config
set_config(display='diagram')  

tda_union

FeatureUnion(n_jobs=-1,
             transformer_list=[('pipeline-1',
                                Pipeline(steps=[('binarizer',
                                                 Binarizer(n_jobs=-1,
                                                           threshold=0.3)),
                                                ('radialfiltration',
                                                 RadialFiltration(center=array([64, 27]),
                                                                  n_jobs=-1)),
                                                ('cubicalpersistence',
                                                 CubicalPersistence(n_jobs=-1)),
                                                ('scaler', Scaler(n_jobs=-1)),
                                                ('featureunion',
                                                 FeatureUnion(transformer_list=[('persistenceentropy',
                                                                                 Persiste...
                                                                                           metric_params={'n_bins': 100,
                                                                                                          'p': 1,
                                                                                                          'sigma': 1.6},
                                                                                           n_jobs=-1)),
                                                                                ('amplitude-11',
                                                                                 Amplitude(metric='heat',
                                                                                           metric_params={'n_bins': 100,
                                                                                                          'p': 1,
                                                                                                          'sigma': 3.2},
                                                                                           n_jobs=-1)),
                                                                                ('amplitude-12',
                                                                                 Amplitude(metric='heat',
                                                                                           metric_params={'n_bins': 100,
                                                                                                          'p': 2,
                                                                                                          'sigma': 1.6},
                                                                                           n_jobs=-1)),
                                                                                ('amplitude-13',
                                                                                 Amplitude(metric='heat',
                                                                                           metric_params={'n_bins': 100,
                                                                                                          'p': 2,
                                                                                                          'sigma': 3.2},
                                                                                           n_jobs=-1))]))]))])

It's now a simple matter to run the whole pipeline:

In [18]:
X_train_tda = tda_union.fit_transform(X_train)
X_train_tda.shape

(4569, 280)

## Training a classifier

In [19]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()
rf.fit(X_train_tda, y_train)

X_test_tda = tda_union.transform(X_test)
rf.score(X_test_tda, y_test)

0.7742782152230971

## Using hyperparameter search with topological pipelines 

In the above pipeline, we can think of our choices for the directions and centers of the filtrations as hyperparameter. To wrap up our analysis, let's see how we can run a hyperparameter search over the directions of the height filtration. We'll use a simplified pipeline to show the main steps, but note that a realistic application would involve running the search over a pipeline like the one in the previous section.

As usual, we define our pipeline in terms of topological transformers and an estimator as the final step:

In [37]:
# Radial filtration pipeline for brain tumor classification
radial_pipeline = Pipeline([
    ('binarizer', Binarizer(threshold=0.3)),  # Adjusted for MRI
    ('filtration', RadialFiltration(center=np.array([64, 64]))),  # Center of 128x128 image
    ('diagram', CubicalPersistence()),
    ('feature', PersistenceEntropy(nan_fill_value=-1)),
    ('classifier', RandomForestClassifier(random_state=42))
])

Next we can search for the best combination of directions, homology dimensions, and number of trees in our Random Forest as follows:

In [ ]:
from sklearn.model_selection import GridSearchCV

center_list = [
    [64, 64],   
    [64, 32],  
    [64, 96],  
    [32, 64],    
    [96, 64],  
]

threshold_list = [0.3]

homology_dimensions_list = [[0, 1]]

n_estimators_list = [100, 200]

param_grid = {
    "binarizer__threshold": threshold_list,
    "filtration__center": [np.array(center) for center in center_list],
    "diagram__homology_dimensions": homology_dimensions_list,
    "classifier__n_estimators": n_estimators_list,
}

grid_search = GridSearchCV(
    estimator=radial_pipeline, param_grid=param_grid, cv=3, n_jobs=-1, verbose=2
)

grid_search.fit(X_train, y_train)

By looking at the best hyperparameters

In [ ]:
grid_search.best_params_

we see that the direction [1, 0] with homology dimension 0 produces the best features. By comparing say a "6" and "9" digit, can you think of a reason why this might be the case?